In [ ]:
%env AWS_PROFILE=kmnlp

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
import math
import re
from os.path import join

import fsspec
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import seaborn as sns
from datetime import datetime
from pydantic import ValidationError

from llm_agent.evaluation.base_eval import EvalReport
from llm_agent.evaluation.utils import SummaryStats
from llm_agent.utils import model_from_file

In [ ]:
sns.reset_defaults()
sns.set_theme(style="whitegrid", palette=None)

In [ ]:
evals_root = "s3://llm-task-evals-897abd67-68fa-4724-a9df-ad3b70ef282c/"

---

In [ ]:
def get_task_eval_uris(evals_root: str, task_id: str) -> list[str]:
    evals_root = join("s3://llm-task-evals-897abd67-68fa-4724-a9df-ad3b70ef282c/", task_id)
    fs, evals_root = fsspec.url_to_fs(evals_root)
    return [fs.unstrip_protocol(path) for path in fs.glob(join(evals_root, "**/aggregated.json"))]

In [ ]:
uri_regex = re.compile(r"^(?:(?P<protocol>[^:/]+)://)?(?P<root>[\w\d\-_/]+)/(?P<task_id>[^/]+)/dataset/(?P<dataset_id>[^/]+)/version/(?P<dataset_version>[^/]+)/(?P<exp_name>[^/]+)/(?P<timestamp>\d{4}-\d{2}-\d{2}T\d{4})(?:_(?P<tag>[^/]+))?/aggregated.json")

def extract_metrics(aggregated_metrics: dict[str, int | float | SummaryStats]) -> dict[str, int | float]:
    out: dict[str, int | float] = {}
    for k, v in aggregated_metrics.items():
        try:
            stats = SummaryStats.model_validate(v)
            out[f"{k}_mean"] = stats.mean
            out[f"{k}_stderr"] = stats.std / math.sqrt(len(stats.values))
        except ValidationError:
            out[k] = v
    return out


def parse_uri(uri: str) -> dict[str, str]:
    match = uri_regex.fullmatch(uri)
    if match is None:
        raise ValueError
    parsed = match.groupdict()
    parsed.pop("protocol")
    parsed.pop("root")
    parsed["timestamp"] = datetime.strptime(parsed["timestamp"], "%Y-%m-%dT%H%M")
    return parsed
    

def extract_record(uri: str) -> dict[str, str | int | float]:
    exp_info = parse_uri(uri)

    eval_report: EvalReport = model_from_file(uri, EvalReport)
    assert exp_info["dataset_id"] == eval_report.experiment_config.dataset_id
    assert exp_info["dataset_version"] == eval_report.experiment_config.dataset_version
    assert exp_info["exp_name"] == eval_report.experiment_config.name
    
    metrics = extract_metrics(eval_report.aggregated_metrics)
    return {**exp_info, **metrics}


def get_records_for_task(evals_root: str, task_id: str) -> dict[str, str | int | float]:
    uris = get_task_eval_uris(evals_root, task_id)
    return [extract_record(uri) for uri in uris]

In [ ]:
def plot_metrics(df: pd.DataFrame, ncols: int = 3):
    metric_cols = [c[:-len("_mean")] for c in df.columns if c.endswith("_mean")]
    plt.close("all")
    nplots = len(metric_cols)
    nrows = math.ceil(nplots / ncols)
    fig, axs = plt.subplots(nrows=nrows, ncols=ncols, squeeze=False, sharex=True, figsize=(ncols*4, nrows*4))
    for i, (mcol, ax) in enumerate(zip(metric_cols, axs.flat)):
        mean_col = f"{mcol}_mean"
        stderr_col = f"{mcol}_stderr"
        for g, gdf in df.groupby("exp_name"):
            ax.plot(gdf["timestamp"], gdf[mean_col], marker="o", label=g)
            ax.fill_between(
                gdf["timestamp"],
                gdf[mean_col] - gdf[stderr_col] * 1.96,
                gdf[mean_col] + gdf[stderr_col] * 1.96,
                alpha=0.2
            )

        ax.set_xlabel("timestamp")
        ax.set_ylabel("")
        if mcol.startswith("pct_"):
            ax.set_ylim((-5, 105))
        else:
            ax.set_ylim((-1, None))
        ax.set_title(mcol)
        legend = ax.legend(title="Experiment", fontsize=9, title_fontsize=10)
        legend.remove()

    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, ncols=min(3, len(labels)), loc="upper center", bbox_to_anchor=(0.5, 1.05))

    for ax in axs.flat:
        # Text in the x-axis will be displayed in 'YYYY-mm' format.
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H'))
        # Rotates and right-aligns the x labels so they don't crowd each other.
        for label in ax.get_xticklabels(which='major'):
            label.set(rotation=45, horizontalalignment='right')

    if nrows == 1:
        for ax in axs.flat[i + 1: ]:
            ax.axis("off")
    else:
        for ax in axs.flat[i + 1: ]:
            # Hide all spines
            for spine in ax.spines.values():
                spine.set_visible(False)
            
            # Hide y-axis ticks and labels
            ax.yaxis.set_ticks([])
            ax.yaxis.set_ticklabels([])
            ax.grid(False)

    fig.tight_layout()

    plt.show()

---

## Algorithm Agent

In [ ]:
df_alg = pd.DataFrame.from_records(get_records_for_task(evals_root, "algorithm_agent"))
df_alg

In [ ]:
plot_metrics(df_alg)

---

## Dataset Picking Agent

In [ ]:
df_dp = pd.DataFrame.from_records(get_records_for_task(evals_root, "dataset_picking_agent"))
df_dp

In [ ]:
plot_metrics(df_dp)

## Dataset Picking Agent - Single Turn

In [ ]:
df_dpst = pd.DataFrame.from_records(get_records_for_task(evals_root, "dataset_picking_agent_single_turn"))
df_dpst

In [ ]:
plot_metrics(df_dpst)